# 04 · Mô hình pretrained kiến trúc Encoder–Decoder

| Mã | Mô hình (Hugging Face) | Đặc điểm | Thí nghiệm |
|---|---|---|---|
| `marian` | Helsinki-NLP/opus-mt-en-vi | MarianMT ~75M, chuyên dịch En→Vi (dữ liệu OPUS) | zero-shot + fine-tune |
| `envit5` | VietAI/envit5-translation | T5 (VietAI), song ngữ Anh–Việt, huấn luyện trên MTet + PhoMT | zero-shot + fine-tune |
| `nllb600m` | facebook/nllb-200-distilled-600M | Đa ngữ 200 ngôn ngữ (Meta) | zero-shot |
| `mt5small` | google/mt5-small | Pretrained đa ngữ nhưng **chưa học dịch** (chỉ span-corruption) | fine-tune |

**Lưu ý khi viết báo cáo:** PhoMT/MTet có chứa phụ đề TED, nên `envit5` có thể đã "thấy" câu gần giống tập test
IWSLT (nhiễm dữ liệu — *data contamination*) → điểm cao cần diễn giải thận trọng.

**Các điểm kỹ thuật đã xử lý:** Marian bắt buộc tiền tố `>>vie<<`; NLLB cần mã ngôn ngữ `eng_Latn → vie_Latn`;
envit5 cần tiền tố `en:` và sinh ra `vi: ...` (được cắt bỏ); họ T5 bị tràn số với fp16 nên trên GPU T4 tự chuyển sang fp32;
tương thích `transformers` 4.x và 5.x.

> **Quy trình chống lỗi:** luôn chạy với `DRY_RUN = True` trước (vài phút, dữ liệu và mô hình tí hon, đi qua **mọi**
> dòng code). Khi không còn lỗi nào, đổi `DRY_RUN = False` rồi chọn **Runtime → Run all** để chạy thật.
> Kết quả chạy thử nằm riêng trong thư mục `dryrun/`, không lẫn với kết quả thật.

> **Nếu Colab mất kết nối / hết phiên:** mở lại notebook, **Run all**. Huấn luyện tự tiếp tục từ checkpoint gần nhất
> (lưu mỗi vài trăm bước trên Drive). Bản dịch đã sinh từ đúng checkpoint sẽ không bị dịch lại.

In [1]:
#@title ⚙️ Tham số
DRY_RUN = False
BEAM = 4
# Bật/tắt từng mô hình và thí nghiệm ở đây
MODELS = {
    "marian":   dict(model_id="Helsinki-NLP/opus-mt-en-vi",
                     spec=dict(family="marian", src_prefix=">>vie<< "),
                     zero_shot=True, finetune=True, last_on_drive=True,
                     ft=dict(lr=5e-5, epochs=5, max_tokens=4096, grad_accum=2, warmup_steps=500, schedule="linear")),
    "envit5":   dict(model_id="VietAI/envit5-translation",
                     spec=dict(family="t5", src_prefix="en: ", tgt_prefix="vi: ", fp16_ok=False),
                     zero_shot=True, finetune=True, last_on_drive=False,
                     ft=dict(lr=1e-4, epochs=3, max_tokens=4096, grad_accum=2, warmup_steps=500, schedule="linear")),
    "nllb600m": dict(model_id="facebook/nllb-200-distilled-600M",
                     spec=dict(family="nllb", src_lang="eng_Latn", tgt_lang="vie_Latn"),
                     zero_shot=True, finetune=False),
    "mt5small": dict(model_id="google/mt5-small",
                     spec=dict(family="t5", src_prefix="translate English to Vietnamese: ", fp16_ok=False),
                     zero_shot=False, finetune=True, last_on_drive=False,
                     ft=dict(lr=5e-4, epochs=6, max_tokens=2048, grad_accum=4, warmup_steps=1000, schedule="linear")),
}
DEV_EVAL_SIZE = 500           # số câu dev dùng để chọn checkpoint khi fine-tune (greedy)
TIME_BUDGET_HOURS = None

In [2]:
#@title 🔧 Kết nối Google Drive + nạp thư viện `envi_lib`
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
PROJECT_DIR = os.environ.get("ENVI_PROJECT_DIR", "/content/drive/MyDrive/NLP_Final_EnVi")
if not os.path.exists(os.path.join(PROJECT_DIR, "envi_lib", "__init__.py")):
    raise FileNotFoundError(f"Chưa có thư viện ở {PROJECT_DIR}/envi_lib — hãy chạy notebook 00_cai_dat.ipynb trước.")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
from envi_lib.common import (Paths, set_seed, get_device, pick_amp_dtype, autocast_ctx, gpu_info, peak_gpu_mem_gb,
                             reset_peak_mem, free_memory, count_params, disk_free_gb, save_predictions,
                             prediction_is_current, save_json, load_json, read_lines, write_lines, fmt_seconds,
                             bf16_supported, check_lib_version, now_str)
check_lib_version("1.0.0")
paths = Paths(PROJECT_DIR, dry_run=DRY_RUN).make_all()
DEVICE = get_device()
print(f"📁 {paths.root}   |   DRY_RUN = {DRY_RUN}   |   thiết bị: {gpu_info()}")

Mounted at /content/drive
📁 /content/drive/MyDrive/NLP_Final_EnVi   |   DRY_RUN = False   |   thiết bị: {'cuda': True, 'torch': '2.11.0+cu130', 'name': 'NVIDIA A100-SXM4-80GB', 'total_gb': 79.3, 'bf16': True}


In [3]:
%pip install -q sacrebleu sacremoses sentencepiece protobuf
import math, time, torch, pandas as pd, transformers
from functools import partial
from envi_lib import data as D, metrics as M, hf_utils as H
from envi_lib.train import TrainConfig, Trainer
AMP = pick_amp_dtype("auto")
print("transformers", transformers.__version__, "| GPU:", gpu_info(), "| autocast:", AMP)
if DEVICE.type != "cuda" and not DRY_RUN:
    raise RuntimeError("Cần GPU: Runtime → Change runtime type → GPU")
data = D.load_all_clean(paths, DRY_RUN)
train_src, train_tgt = data["train"]; dev_src, dev_ref = data["dev"]; test_src, test_ref = data["test"]
dev_n = 50 if DRY_RUN else DEV_EVAL_SIZE
print(f"train {len(train_src):,} | dev {len(dev_src):,} (dùng {dev_n} câu khi fine-tune) | test {len(test_src):,}")
LOCAL_CKPT = "/content/ckpt_local" if IN_COLAB else str(paths.work / "ckpt_local")

ℹ️  Phát hiện torchao bản cũ của Colab — đã báo peft bỏ qua torchao (không dùng tới trong đồ án).
transformers 5.20.1 | GPU: {'cuda': True, 'torch': '2.11.0+cu130', 'name': 'NVIDIA A100-SXM4-80GB', 'total_gb': 79.3, 'bf16': True} | autocast: torch.bfloat16
train 130,400 | dev 1,500 (dùng 500 câu khi fine-tune) | test 1,268


In [4]:
#@title 🧪 Kiểm tra trước (preflight): tải từng mô hình, kiểm tra token đặc biệt, dịch thử 3 câu
for key, cfg in MODELS.items():
    if not (cfg.get("zero_shot") or cfg.get("finetune")):
        continue
    spec = H.Seq2SeqSpec(**cfg["spec"])
    tok, model = H.load_seq2seq(cfg["model_id"], spec, DEVICE)
    out = H.hf_seq2seq_translate(model, tok, spec, test_src[:3], beam_size=2, amp_dtype=AMP, progress=False)
    print(f"✔ {key:9s} {count_params(model) / 1e6:6.0f}M tham số | ví dụ: {out[0][:80]!r}")
    del tok, model; free_memory()

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  289MB            

... (42 outputs left out)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✔ mt5small     300M tham số | ví dụ: '<extra_id_0>.'


In [5]:
#@title 🚀 Zero-shot và fine-tune
def translate_test(key, system, model, tok, spec, fp):
    if prediction_is_current(paths, system, fp, len(test_ref)):
        return
    reset_peak_mem(); t0 = time.time()
    hyps = H.hf_seq2seq_translate(model, tok, spec, test_src, beam_size=BEAM, batch_size=64, amp_dtype=AMP)
    dt = time.time() - t0
    save_predictions(paths, system, hyps, expected_n=len(test_ref), meta={
        "fingerprint": fp, "architecture": "Encoder-Decoder", "model_id": MODELS[key]["model_id"],
        "training": fp["mode"], "params": count_params(model), "decode_time_s": dt, "beam": BEAM,
        "peak_mem_gb": peak_gpu_mem_gb(), **fp})
    print(f"   {system}: test BLEU = {M.quick_bleu(hyps, test_ref):.2f}  [{dt:.0f}s]")

summary = {}
for key, cfg in MODELS.items():
    spec = H.Seq2SeqSpec(**cfg["spec"])
    print(f"\n{'=' * 80}\n▶ {key}: {cfg['model_id']}\n{'=' * 80}")
    # ---------------- zero-shot
    if cfg.get("zero_shot"):
        tok, model = H.load_seq2seq(cfg["model_id"], spec, DEVICE)
        translate_test(key, f"{key}_zeroshot", model, tok, spec, {"mode": "Pretrained (zero-shot)", "beam": BEAM})
        del tok, model; free_memory()
    # ---------------- fine-tune
    if cfg.get("finetune"):
        tok, model = H.load_seq2seq(cfg["model_id"], spec, DEVICE, for_training=True)
        feats, n_trunc = H.make_hf_seq2seq_features(tok, spec, train_src, train_tgt, max_len=256)
        print(f"   {len(feats):,} cặp câu huấn luyện ({n_trunc} cặp bị cắt ở 256 token)")
        ft = dict(cfg["ft"])
        amp_mode = "auto" if (spec.fp16_ok or bf16_supported()) else "fp32"
        if DRY_RUN:
            ft.update(epochs=1, warmup_steps=5, max_tokens=1024, grad_accum=1)
        tcfg = TrainConfig(**ft, label_smoothing=0.1, patience=2, amp=amp_mode, save_every_steps=20 if DRY_RUN else 500,
                           log_every_steps=10 if DRY_RUN else 100, max_steps=20 if DRY_RUN else None,
                           weight_decay=0.01, time_budget_hours=TIME_BUDGET_HOURS)
        ck = paths.checkpoints / f"{key}_ft"
        last_dir = ck if cfg.get("last_on_drive", True) else os.path.join(LOCAL_CKPT, f"{key}_ft")
        eval_amp = AMP if (spec.fp16_ok or AMP != torch.float16) else None

        def eval_fn(m, tok=tok, spec=spec):
            hy = H.hf_seq2seq_translate(m, tok, spec, dev_src[:dev_n], beam_size=1, batch_size=64,
                                        amp_dtype=eval_amp, progress=False)
            return {"score": M.quick_bleu(hy, dev_ref[:dev_n])}

        def save_best(m, d, tok=tok):
            m.save_pretrained(d); tok.save_pretrained(d)

        trainer = Trainer(model, H.FeatDataset(feats), partial(H.hf_seq2seq_collate, pad_id=tok.pad_token_id),
                          [max(len(a), len(b)) for a, b in feats], tcfg, ck, H.hf_seq2seq_loss, eval_fn, device=DEVICE,
                          save_best_fn=save_best, run_name=f"{key}_ft", last_dir=last_dir,
                          extra_meta={"model_id": cfg["model_id"]})
        if not trainer.last_path.exists():
            trainer.evaluate(tag="trước fine-tune")   # điểm dev của mô hình gốc (để thấy fine-tune cải thiện bao nhiêu)
        trainer.train()
        summary[key] = {"best_dev_bleu": trainer.state["best_score"], "train_time_s": trainer.state["train_seconds"]}
        if not trainer.state["finished"]:
            print("⏸ Chưa xong (giới hạn thời gian) — chạy lại ô này để tiếp tục."); break
        best_step = trainer.state["best_step"]; train_s = trainer.state["train_seconds"]
        del trainer, model; free_memory()
        tok_b, model_b = H.load_seq2seq(str(ck / "best"), spec, DEVICE)
        fp = {"mode": "Pretrained + fine-tune", "beam": BEAM, "ckpt_step": best_step, "train_time_s": train_s}
        translate_test(key, f"{key}_ft", model_b, tok_b, spec, fp)
        del tok, tok_b, model_b; free_memory()
save_json(summary, paths.results / "04_summary.json")
summary


▶ marian: Helsinki-NLP/opus-mt-en-vi


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

✔ marian_zeroshot: đã có bản dịch từ đúng checkpoint này — bỏ qua (xoá file trong predictions/ nếu muốn dịch lại).


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

... (54 outputs left out)

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Dịch (beam=4):   0%|          | 0/40 [00:00<?, ?it/s]

✅ Đã lưu 1268 bản dịch: /content/drive/MyDrive/NLP_Final_EnVi/predictions/mt5small_ft.vi.txt
   mt5small_ft: test BLEU = 26.65  [56s]


{'marian': {'best_dev_bleu': 36.23298479362337,
  'train_time_s': 329.5627131462097},
 'envit5': {'best_dev_bleu': 33.49871299696942,
  'train_time_s': 876.2414994239807},
 'mt5small': {'best_dev_bleu': 25.604580809787567,
  'train_time_s': 2848.7113535404205}}

In [6]:
#@title 📋 Tóm tắt nhanh (đánh giá đầy đủ ở notebook 06)
rows = []
for key in MODELS:
    for f in sorted(paths.predictions.glob(f"{key}_*.vi.txt")):
        system = f.name.replace(".vi.txt", ""); meta = load_json(paths.predictions / f"{system}.meta.json")
        rows.append({"hệ thống": system, "chế độ": meta.get("training"), "tham số (M)": round(meta["params"] / 1e6),
                     "BLEU test": round(M.quick_bleu(read_lines(f), test_ref), 2),
                     "thời gian dịch (s)": round(meta["decode_time_s"], 1)})
pd.DataFrame(rows)

,hệ thống,chế độ,tham số (M),BLEU test,thời gian dịch (s)
0,marian_ft,Pretrained + fine-tune,72,37.50,17.1
1,marian_zeroshot,Pretrained (zero-shot),72,36.66,16.5
2,envit5_ft,Pretrained + fine-tune,275,33.65,67.0
3,envit5_zeroshot,Pretrained (zero-shot),275,13.27,50.7
4,nllb600m_zeroshot,Pretrained (zero-shot),615,34.16,29.7
5,mt5small_ft,Pretrained + fine-tune,300,26.65,55.5


In [7]:
#@title 🧹 (Tuỳ chọn) Xoá checkpoint "last" để giải phóng Drive — CHỈ chạy khi mọi fine-tune đã xong
CLEAN_LAST = False
if CLEAN_LAST:
    for key in MODELS:
        for p in [paths.checkpoints / f"{key}_ft" / "last.pt"]:
            if p.exists():
                print("xoá", p); p.unlink()